In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
from src.data_preparation import OCTMNISTDataPreprocessor
preprocessor=OCTMNISTDataPreprocessor(data_path="../data/octmnist_64.npz")
X_train,X_val, X_test, y_train, y_val, y_test = preprocessor.load_dataset()

Loading OCTMNIST data...
Original splits loaded: Training Datapoints: 97477, Validation Datapoints: 10832, Test Datapoints: 1000
Total Data Points: 109309
Creating new stratified splits...
New splits created: Training Datapoints: 76516, Validation Datapoints: 16396, Test Datapoints: 16397
Shapes of returned Splits: X_train: (76516, 64, 64, 1), X_val: (16396, 64, 64, 1), X_test: (16397, 64, 64, 1)
Shapes of returned Labels: y_train: (76516,), y_val: (16396,), y_test: (16397,)


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

class OCT(Dataset):
    def __init__(self, X, y):
        X = np.asarray(X).squeeze()              # (N, 64, 64)
        if X.max() > 1.0:                        # make sure values are in [0, 1]
            X = X / 255.0
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(np.asarray(y).ravel(), dtype=torch.long)
        self.norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        img = self.X[i].unsqueeze(0).repeat(3, 1, 1)   # grayscale -> 3 channels
        return self.norm(img), self.y[i]

train_dl = DataLoader(OCT(X_train, y_train), batch_size=128, shuffle=True)
val_dl   = DataLoader(OCT(X_val,   y_val),   batch_size=128)
test_dl  = DataLoader(OCT(X_test,  y_test),  batch_size=128)

In [ ]:
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Conv2d(512, 4, kernel_size=1)   # 1000 classes -> 4
model.num_classes = 4
model = model.to(device)